# Un agente no alcanza: poné a trabajar a todo un equipo

**Student Builder Group · Universidad Austral · 20/11/2026**

Un solo agente con muchas tools funciona... hasta que deja de funcionar: se confunde, mezcla tareas y es difícil de mantener.
La solución es la misma que en un equipo de personas: **dividir el trabajo entre especialistas**.

En este notebook vemos los **5 patrones multi-agente** de [Strands Agents](https://strandsagents.com) con un ejemplo que todos entendemos: **organizar un viaje**.

| Patrón | ¿Quién decide qué pasa después? | Analogía |
|---|---|---|
| **Agents as Tools** | Un agente "jefe" | Un jefe que le pide cosas a sus especialistas |
| **Swarm** | Cada agente le pasa la posta al que sigue | Un grupo de TP que se reparte el trabajo solo |
| **Graph** | Vos, en código | Una línea de montaje |
| **Workflow** | Un DAG fijo de tareas (con paralelismo) | Una receta: algunas cosas se hacen a la vez |
| **A2A** | Un protocolo entre agentes remotos | Llamar a otra empresa para que haga su parte |

> Todas las tools usan **datos inventados**: no hace falta ninguna API externa ni API key. Solo necesitás acceso a un modelo (usamos Amazon Bedrock).

---
## 0. Setup

Requisitos:
- Python 3.10 o superior.
- Credenciales de AWS con acceso a Amazon Bedrock. Si usás un perfil con nombre, ponelo en `AWS_PROFILE`. Si usás otro proveedor (Anthropic, OpenAI, Ollama...), cambiá solamente la función `modelo()` de abajo.

Instalamos un solo paquete: Strands, con el extra `a2a` para el último patrón.

In [ ]:
%pip install -q "strands-agents[a2a]"

In [ ]:
import os
import boto3
from strands import Agent, tool
from strands.models.bedrock import BedrockModel

MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"  # rápido y barato; podés cambiarlo por uno más grande
REGION = "us-east-1"
AWS_PROFILE = None  # None = perfil por defecto; o poné el nombre de tu perfil, ej. "mi-perfil"

# Lo dejamos en el entorno para que cualquier cliente de AWS (y el servidor A2A, que hereda el entorno) use este perfil
if AWS_PROFILE:
    os.environ["AWS_PROFILE"] = AWS_PROFILE
SESION = boto3.Session(profile_name=AWS_PROFILE, region_name=REGION)
print("Cuenta AWS:", SESION.client("sts").get_caller_identity()["Account"])


def modelo():
    # Cada agente recibe su propia instancia del modelo
    return BedrockModel(model_id=MODEL_ID, boto_session=SESION)


# Probamos que todo ande. callback_handler=None = no imprimir mientras "piensa", solo el resultado final.
agente = Agent(model=modelo(), callback_handler=None)
print(agente("Saludá a los estudiantes de la Austral en una línea."))

---
## 1. Nuestras tools (de mentira)

Dos tools muy simples que devuelven datos inventados. Cada una imprime un 🔧 cuando la llaman, así vemos **qué agente hace qué**.

In [ ]:
VUELOS = {
    "bariloche": [
        {"aerolinea": "Aerolíneas Argentinas", "precio_usd": 120, "sale": "08:00"},
        {"aerolinea": "Flybondi", "precio_usd": 85, "sale": "13:30"},
        {"aerolinea": "JetSMART", "precio_usd": 95, "sale": "19:10"},
    ],
    "mendoza": [
        {"aerolinea": "Aerolíneas Argentinas", "precio_usd": 90, "sale": "07:15"},
        {"aerolinea": "JetSMART", "precio_usd": 70, "sale": "21:40"},
    ],
}

CLIMA = {
    "bariloche": {"temp_max_c": 8, "lluvia": "60%", "detalle": "frío y con chances de nieve"},
    "mendoza": {"temp_max_c": 27, "lluvia": "5%", "detalle": "soleado y seco"},
}


@tool
def buscar_vuelos(destino: str) -> list:
    '''Busca vuelos desde Buenos Aires hacia un destino de Argentina.

    Args:
        destino: ciudad de destino, por ejemplo "Bariloche".
    '''
    print(f"   🔧 buscar_vuelos({destino!r})")
    return VUELOS.get(destino.lower(), [])


@tool
def ver_clima(ciudad: str) -> dict:
    '''Devuelve el pronóstico del fin de semana para una ciudad de Argentina.

    Args:
        ciudad: nombre de la ciudad, por ejemplo "Mendoza".
    '''
    print(f"   🔧 ver_clima({ciudad!r})")
    return CLIMA.get(ciudad.lower(), {"error": "ciudad no encontrada"})


PEDIDO = "Quiero ir de Buenos Aires a Bariloche este finde. ¿Qué vuelo me conviene y qué ropa llevo?"

---
## 2. Agents as Tools: un jefe y sus especialistas

```
                 ┌──> agente_vuelos  (buscar_vuelos)
jefe_de_viaje ───┤
                 └──> agente_clima   (ver_clima)
```

Un agente **orquestador** usa a otros agentes **como si fueran tools**. El jefe no sabe nada de vuelos ni de clima: solo sabe a quién preguntarle.

- ✅ Simple, todo en el mismo proceso, el jefe tiene el control.
- 🤔 Usalo cuando tenés un "agente principal" que necesita ayuda especializada.

In [ ]:
agente_vuelos = Agent(
    name="agente_vuelos",
    model=modelo(),
    tools=[buscar_vuelos],
    system_prompt="Sos experto en vuelos. Buscá opciones y recomendá la más barata. Respondé en 2 líneas.",
    callback_handler=None,
)

agente_clima = Agent(
    name="agente_clima",
    model=modelo(),
    tools=[ver_clima],
    system_prompt="Sos experto en clima. Decí cómo va a estar y qué ropa llevar. Respondé en 2 líneas.",
    callback_handler=None,
)

jefe = Agent(
    name="jefe_de_viaje",
    model=modelo(),
    tools=[
        # .as_tool() convierte un agente en una tool
        agente_vuelos.as_tool(description="Experto en vuelos: busca y recomienda vuelos."),
        agente_clima.as_tool(description="Experto en clima: pronóstico y qué ropa llevar."),
    ],
    system_prompt=(
        "Sos un organizador de viajes para estudiantes. No sabés de vuelos ni de clima: "
        "delegá en tus especialistas y armá una respuesta corta en español."
    ),
    callback_handler=None,
)

respuesta = jefe(PEDIDO)
print("\n" + str(respuesta))

---
## 3. Swarm: el equipo se organiza solo

```
agente_vuelos  <──>  agente_clima  <──>  agente_resumen
        (se pasan la posta con handoff_to_agent)
```

En un **Swarm** no hay jefe. Le damos la tarea al primer agente y **cada uno decide a quién pasarle el trabajo** (Strands les agrega automáticamente una tool `handoff_to_agent`).

- ✅ Flexible: no tenés que definir el orden.
- 🤔 Menos predecible: el recorrido puede cambiar entre ejecuciones.

In [ ]:
from strands.multiagent import Swarm


def equipo_de_viaje(en_swarm=False):
    # Creamos agentes nuevos (sin memoria de ejemplos anteriores)
    extra = " Después pasale la posta a quien corresponda; nunca le respondas vos al usuario." if en_swarm else ""
    vuelos = Agent(name="vuelos", model=modelo(), tools=[buscar_vuelos], callback_handler=None,
                   system_prompt="Sos experto en vuelos. Buscá vuelos al destino pedido y elegí el más barato." + extra)
    clima = Agent(name="clima", model=modelo(), tools=[ver_clima], callback_handler=None,
                  system_prompt="Sos experto en clima. Consultá el clima del destino pedido." + extra)
    resumen = Agent(name="resumen", model=modelo(), callback_handler=None,
                    system_prompt="Armás la respuesta final para el estudiante: corta, clara y en español.")
    return vuelos, clima, resumen


def mostrar(resultado, recorrido):
    # Muestra qué agentes trabajaron (en orden) y la respuesta del último
    print("\nRecorrido:", " → ".join(nodo.node_id for nodo in recorrido))
    ultimo = recorrido[-1].node_id
    print("\n" + str(resultado.results[ultimo].result))


vuelos, clima, resumen = equipo_de_viaje(en_swarm=True)
swarm = Swarm([vuelos, clima, resumen], max_handoffs=6)  # el primero de la lista recibe el pedido

resultado = swarm(PEDIDO)
mostrar(resultado, resultado.node_history)

---
## 4. Graph: vos definís el camino

```
vuelos ──> clima ──> resumen
```

En un **Graph** cada agente es un **nodo** y vos definís las **flechas** (edges) en código. Cada nodo recibe el pedido original + lo que hicieron los nodos anteriores.

- ✅ Predecible y fácil de auditar: siempre el mismo orden.
- 🤔 Menos flexible: si el problema cambia, cambiás el código.
- 💡 Las flechas pueden tener **condiciones** (ej: "ir a `resumen` solo si hay vuelos").

In [ ]:
from strands.multiagent import GraphBuilder

vuelos, clima, resumen = equipo_de_viaje()

builder = GraphBuilder()
builder.add_node(vuelos, "vuelos")
builder.add_node(clima, "clima")
builder.add_node(resumen, "resumen")
builder.add_edge("vuelos", "clima")    # después de vuelos → clima
builder.add_edge("clima", "resumen")   # después de clima → resumen
builder.set_entry_point("vuelos")
builder.set_execution_timeout(120)   # por las dudas, un límite de tiempo
grafo = builder.build()

resultado = grafo(PEDIDO)
mostrar(resultado, resultado.execution_order)

---
## 5. Workflow: tareas con dependencias (y en paralelo)

```
[ vuelos ] ──┐
             ├──> [ resumen ]     (resumen espera a que terminen las dos)
[ clima  ] ──┘
```

Un **Workflow** es una lista fija de **tareas** con **dependencias** (un DAG). El motor se encarga de:
- correr **en paralelo** las tareas que no dependen entre sí (vuelos y clima),
- y arrancar una tarea **recién cuando terminaron todas sus dependencias** (resumen).

**¿Y la diferencia con Graph?** En el Graph armamos los agentes y las flechas a mano. En un Workflow describimos las tareas **como datos** (una lista) y el motor arma el flujo solo.

> En versiones anteriores, `strands-agents-tools` traía una tool `workflow`. Hoy Strands recomienda armarlo sobre el mismo `GraphBuilder`, que es lo que hacemos acá en ~20 líneas.

In [ ]:
# 1) Las tareas, definidas como datos
TAREAS = [
    {"id": "vuelos", "tools": [buscar_vuelos], "tarea": "Buscá vuelos al destino pedido y elegí el más barato. 1 línea."},
    {"id": "clima", "tools": [ver_clima], "tarea": "Consultá el clima del destino pedido. 1 línea."},
    {"id": "resumen", "depende_de": ["vuelos", "clima"],
     "tarea": "Con los resultados de las tareas anteriores, recomendá vuelo y ropa en 3 líneas, en español."},
]


# 2) Un mini motor de workflows
def terminaron(dependencias):
    # Condición: solo avanzar cuando TODAS las dependencias están completas
    def condicion(estado):
        listas = {nodo.node_id for nodo in estado.completed_nodes}
        return all(dep in listas for dep in dependencias)
    return condicion


def crear_workflow(tareas):
    builder = GraphBuilder()
    for t in tareas:
        agente = Agent(name=t["id"], model=modelo(), tools=t.get("tools", []),
                       system_prompt=t["tarea"], callback_handler=None)
        builder.add_node(agente, t["id"])
    for t in tareas:
        deps = t.get("depende_de", [])
        for dep in deps:
            builder.add_edge(dep, t["id"], condition=terminaron(deps))
    builder.set_execution_timeout(120)
    return builder.build()  # las tareas sin dependencias arrancan solas (y en paralelo)


# 3) Lo ejecutamos
workflow = crear_workflow(TAREAS)
resultado = workflow(PEDIDO)
mostrar(resultado, resultado.execution_order)

In [ ]:
# Cada tarea queda con su estado y su resultado: útil para auditar o reintentar
for nodo in resultado.execution_order:
    print(f"{nodo.node_id:8} → {nodo.execution_status.value:10} ({nodo.execution_time} ms)")

---
## 6. A2A: agentes que viven en otro lado

```
   este notebook                           otro proceso (puerto 9010+)
┌──────────────────┐   HTTP (protocolo A2A)   ┌───────────────────┐
│  agente_viajes   │ ───────────────────────> │   agente_clima    │
└──────────────────┘                          └───────────────────┘
```

**A2A (Agent-to-Agent)** es un protocolo abierto para que agentes de distintos procesos, equipos, empresas o incluso frameworks hablen entre sí por HTTP.

- El que **ofrece** el agente lo publica con `A2AServer`.
- El que lo **usa** se conecta con `A2AAgent`, y lo usa igual que un agente local.

Para simular "otro equipo", escribimos el agente de clima en un archivo aparte y lo levantamos como **otro proceso**. Si el puerto 9010 está ocupado, usamos el siguiente libre.

In [ ]:
%%writefile servidor_clima.py
# Este archivo representa el agente de OTRO equipo, corriendo en su propio servidor.
import sys
import boto3
from strands import Agent, tool
from strands.models.bedrock import BedrockModel
from strands.multiagent.a2a import A2AServer

CLIMA = {
    "bariloche": {"temp_max_c": 8, "lluvia": "60%", "detalle": "frío y con chances de nieve"},
    "mendoza": {"temp_max_c": 27, "lluvia": "5%", "detalle": "soleado y seco"},
}


@tool
def ver_clima(ciudad: str) -> dict:
    '''Devuelve el pronóstico del fin de semana para una ciudad de Argentina.

    Args:
        ciudad: nombre de la ciudad, por ejemplo "Mendoza".
    '''
    return CLIMA.get(ciudad.lower(), {"error": "ciudad no encontrada"})


agente_clima = Agent(
    name="agente_clima",
    description="Experto en clima de ciudades de Argentina.",
    model=BedrockModel(
        model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0",
        boto_session=boto3.Session(region_name="us-east-1"),  # usa el AWS_PROFILE que hereda del notebook
    ),
    tools=[ver_clima],
    system_prompt="Sos experto en clima. Respondé en 2 líneas, en español.",
    callback_handler=None,
)

# El puerto llega como argumento (por defecto 9010)
PUERTO = int(sys.argv[1]) if len(sys.argv) > 1 else 9010
A2AServer(agent=agente_clima, host="127.0.0.1", port=PUERTO).serve()

In [ ]:
import socket, subprocess, sys, time, urllib.request


def puerto_libre(desde=9010, intentos=20):
    # Probamos puertos desde 9010 hasta encontrar uno que no esté en uso
    for puerto in range(desde, desde + intentos):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(("127.0.0.1", puerto))
                return puerto
            except OSError:
                print(f"Puerto {puerto} ocupado, probando el siguiente...")
    raise RuntimeError(f"No hay puertos libres entre {desde} y {desde + intentos - 1}")


PUERTO = puerto_libre()
URL_A2A = f"http://127.0.0.1:{PUERTO}"

# Levantamos el servidor en otro proceso (como si fuera otro equipo / otro container)
# (sus logs van a servidor_clima.log para no ensuciar el notebook)
log = open("servidor_clima.log", "w")
servidor = subprocess.Popen([sys.executable, "servidor_clima.py", str(PUERTO)], stdout=log, stderr=log)

# Esperamos a que publique su "Agent Card" (la ficha que describe al agente)
for _ in range(60):
    try:
        urllib.request.urlopen(f"{URL_A2A}/.well-known/agent-card.json")
        print(f"✅ Servidor A2A listo en {URL_A2A}")
        break
    except Exception:
        if servidor.poll() is not None:  # el proceso se murió: no tiene sentido seguir esperando
            print("❌ El servidor se cerró: mirá servidor_clima.log")
            break
        time.sleep(1)
else:
    print("❌ El servidor no arrancó: mirá servidor_clima.log")

In [ ]:
from strands.agent.a2a_agent import A2AAgent

# Nos conectamos al agente remoto. A partir de acá se usa igual que un Agent local.
clima_remoto = A2AAgent(endpoint=URL_A2A)

print(clima_remoto("¿Cómo va a estar el clima en Mendoza?"))

In [ ]:
# Y lo podemos combinar con los otros patrones: por ejemplo, darle el agente remoto a un agente local como tool
@tool
def consultar_clima_remoto(pregunta: str) -> str:
    '''Le pregunta al agente de clima del otro equipo (vía A2A).

    Args:
        pregunta: la consulta sobre el clima.
    '''
    print(f"   🌐 A2A → agente_clima: {pregunta!r}")
    return str(clima_remoto(pregunta))


agente_viajes = Agent(
    model=modelo(),
    tools=[buscar_vuelos, consultar_clima_remoto],
    system_prompt="Sos un organizador de viajes para estudiantes. Respondé corto en español.",
    callback_handler=None,
)

print("\n" + str(agente_viajes(PEDIDO)))

In [ ]:
# Apagamos el servidor A2A
servidor.terminate()
log.close()
print("Servidor apagado")

---
## Resumen: ¿cuál uso?

| Si... | Usá |
|---|---|
| Tenés un agente principal que necesita ayuda de especialistas | **Agents as Tools** |
| No sabés de antemano el orden y querés que el equipo se organice solo | **Swarm** |
| El orden importa y querés que siempre sea el mismo | **Graph** |
| Tenés tareas fijas con dependencias y algunas pueden ir en paralelo | **Workflow** |
| El agente lo mantiene otro equipo o corre en otro servicio | **A2A** |

Y se pueden **combinar**: un nodo de un Graph puede ser un agente remoto por A2A, un agente de un Swarm puede tener sub-agentes como tools, etc.

### Para practicar 💪
1. Agregá un `agente_presupuesto` con una tool `calcular_presupuesto(dias, precio_vuelo)` y sumalo al jefe del patrón 1.
2. En el Graph, agregá una flecha condicional: que `resumen` solo corra si hay vuelos para el destino (probá con un destino que no existe).
3. Cambiá el pedido a "Mendoza" y fijate si el Swarm hace el mismo recorrido que con Bariloche.